In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

print("Project root:", PROJECT_ROOT)
print("Python environment is ready!")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

print("\nProject folders:")
for folder in ["data/raw", "data/interim", "data/processed",
               "notebooks", "src", "outputs/reports", "docs"]:
    path = PROJECT_ROOT / folder
    print(f"{folder}: {'OK' if path.exists() else 'MISSING'}")

Project root: c:\Users\Admin\OneDrive\Documents\india-economic-complexity
Python environment is ready!
Pandas version: 3.1.0
NumPy version: 2.5.4

Project folders:
data/raw: OK
data/interim: OK
data/processed: OK
notebooks: OK
src: OK
outputs/reports: OK
docs: OK


In [3]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
raw_folder = project_root / "data" / "raw"

# Find CSV files in data/raw
csv_files = list(raw_folder.glob("*.csv"))

print("CSV files found:", len(csv_files))
for file in csv_files:
    print(f" - {file.name} ({file.stat().st_size / (1024**2):.2f} MB)")

# Find the expected HS92 dataset
matches = [
    file for file in csv_files
    if "HS92" in file.name.upper()
    and "COUNTRY" in file.name.upper()
    and "PRODUCT" in file.name.upper()
]

if not matches:
    print("\nExpected hs92_country_product_year_4.csv not found.")
    print("Check the filename and ensure the file is inside data/raw.")
else:
    file = matches[0]
    print(f"\nInspecting: {file.name}")

    # Read only a small sample; don't load the full dataset
    sample = pd.read_csv(file, nrows=5, low_memory=False)

    print("\nColumn names:")
    print(sample.columns.tolist())

    print("\nFirst 5 rows:")
    display(sample)

    print("\nData types:")
    print(sample.dtypes)

CSV files found: 1
 - hs92_country_product_year_4.csv (430.82 MB)

Inspecting: hs92_country_product_year_4.csv

Column names:
['country_id', 'country_iso3_code', 'product_id', 'product_hs92_code', 'year', 'export_value', 'import_value', 'global_market_share', 'export_rca', 'distance', 'cog', 'pci']

First 5 rows:


,country_id,country_iso3_code,product_id,product_hs92_code,year,export_value,import_value,global_market_share,export_rca,distance,cog,pci
0,4,AFG,650,101,2001,0,13442,0.000000,0.000000,0.915,1.440,0.208
1,4,AFG,650,101,2002,8665,1521,0.000006,0.396980,0.898,1.340,0.432
2,4,AFG,650,101,2003,0,138551,0.000000,0.000000,0.944,1.282,0.437
3,4,AFG,650,101,2004,0,49538,0.000000,0.000000,0.938,1.407,0.214
4,4,AFG,650,101,2005,7054,31038,0.000003,0.159574,0.921,1.248,-0.051



Data types:
country_id               int64
country_iso3_code          str
product_id               int64
product_hs92_code        int64
year                     int64
export_value             int64
import_value             int64
global_market_share    float64
export_rca             float64
distance               float64
cog                    float64
pci                    float64
dtype: object


In [5]:

from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
file = project_root / "data" / "raw" / "hs92_country_product_year_4.csv"

# Inspect the dataset's actual coverage and quality
total_rows = 0
years = set()
missing_counts = {}
invalid_code_rows = 0
code_examples = set()
min_code = None
max_code = None

for chunk in pd.read_csv(file, chunksize=200_000, low_memory=False):
    total_rows += len(chunk)

    years.update(chunk["year"].dropna().astype(int).unique())

    for col in [
        "country_id", "country_iso3_code", "product_id",
        "product_hs92_code", "year", "export_value",
        "import_value", "export_rca", "distance", "cog", "pci"
    ]:
        missing_counts[col] = (
            missing_counts.get(col, 0) + chunk[col].isna().sum()
        )

    codes = chunk["product_hs92_code"].dropna()
    if not codes.empty:
        chunk_min = codes.min()
        chunk_max = codes.max()
        min_code = chunk_min if min_code is None else min(min_code, chunk_min)
        max_code = chunk_max if max_code is None else max(max_code, chunk_max)
        code_examples.update(codes.head(10).astype(int).tolist())

    # Convert product codes to numbers for validation.
# Invalid text values become NaN instead of causing a TypeError.
numeric_codes = pd.to_numeric(
    chunk["product_hs92_code"],
    errors="coerce"
)

invalid_code_rows += (
    numeric_codes.isna().sum()
    + (numeric_codes <= 0).sum()
)

print("Total rows:", f"{total_rows:,}")
print("Year range:", min(years) if years else None, "to",
      max(years) if years else None)
print("Number of years:", len(years))
print("Product code minimum:", min_code)
print("Product code maximum:", max_code)
print("Example product codes:", sorted(code_examples))
print("Rows with missing/non-positive product codes:", invalid_code_rows)

print("\nMissing values by column:")
for col, count in missing_counts.items():
    print(f"{col}: {count:,}")


Total rows: 6,752,528
Year range: 1995 to 2024
Number of years: 30
Product code minimum: 0101
Product code maximum: XXXX
Example product codes: [101, 812, 813, 1508, 1509, 1517, 1602, 2603, 2849, 3209, 3506, 3507, 4202, 4418, 4419, 4809, 4810, 5511, 5512, 5606, 5802, 5803, 5804, 5810, 5811, 5910, 6401, 6506, 6507, 6901, 6908, 7414, 7415, 7804, 7806, 7901, 8204, 8205, 8210, 8311, 8412, 8417, 8418, 8470, 8541, 8542, 8701, 8702, 9008, 9009]
Rows with missing/non-positive product codes: 92

Missing values by column:
country_id: 0
country_iso3_code: 0
product_id: 0
product_hs92_code: 0
year: 0
export_value: 0
import_value: 0
export_rca: 17
distance: 17
cog: 17
pci: 0


In [6]:

from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
file = project_root / "data" / "raw" / "hs92_country_product_year_4.csv"

special_codes = {}

for chunk in pd.read_csv(
    file,
    chunksize=200_000,
    dtype={"product_hs92_code": "string"},
    low_memory=False
):
    codes = chunk["product_hs92_code"].str.strip()

    # A valid four-digit HS code consists of digits only.
    non_numeric = codes.notna() & ~codes.str.fullmatch(r"\d{1,4}")

    counts = codes[non_numeric].value_counts()
    for code, count in counts.items():
        special_codes[code] = special_codes.get(code, 0) + int(count)

print("Non-numeric product codes and their row counts:")
if special_codes:
    for code, count in sorted(special_codes.items()):
        print(f"{code!r}: {count:,} rows")
else:
    print("None found.")

print("\nRows where product_hs92_code is missing:")
missing_count = 0
for chunk in pd.read_csv(
    file,
    chunksize=200_000,
    dtype={"product_hs92_code": "string"},
    low_memory=False
):
    missing_count += chunk["product_hs92_code"].isna().sum()

print(f"{missing_count:,}")


Non-numeric product codes and their row counts:
'XXXX': 3,114 rows

Rows where product_hs92_code is missing:
0


In [7]:

from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
file = project_root / "data" / "raw" / "hs92_country_product_year_4.csv"
output_file = project_root / "data" / "interim" / "hs92_product_code_inventory.csv"

unique_codes = set()

for chunk in pd.read_csv(
    file,
    chunksize=200_000,
    usecols=["product_hs92_code"],
    dtype={"product_hs92_code": "string"},
    low_memory=False
):
    codes = chunk["product_hs92_code"].str.strip().dropna()
    unique_codes.update(codes.unique())

inventory = pd.DataFrame({"product_hs92_code": sorted(unique_codes)})

# Preserve special codes such as XXXX.
inventory["is_numeric_code"] = inventory["product_hs92_code"].str.fullmatch(r"\d+")
inventory["code_4digit"] = pd.NA

numeric = inventory["is_numeric_code"]
inventory.loc[numeric, "code_4digit"] = (
    inventory.loc[numeric, "product_hs92_code"].str.zfill(4)
)

inventory.to_csv(output_file, index=False)

print("Distinct codes found:", len(inventory))
print("Numeric codes:", int(numeric.sum()))
print("Special codes:", int((~numeric).sum()))
print("\nSpecial codes:")
print(inventory.loc[~numeric, "product_hs92_code"].to_list())
print("\nFirst 20 numeric codes:")
print(inventory.loc[numeric, ["product_hs92_code", "code_4digit"]].head(20).to_string(index=False))
print("\nSaved inventory to:", output_file)


Distinct codes found: 1243
Numeric codes: 1242
Special codes: 1

Special codes:
['XXXX']

First 20 numeric codes:
product_hs92_code code_4digit
             0101        0101
             0102        0102
             0103        0103
             0104        0104
             0105        0105
             0106        0106
             0201        0201
             0202        0202
             0203        0203
             0204        0204
             0205        0205
             0206        0206
             0207        0207
             0208        0208
             0209        0209
             0210        0210
             0301        0301
             0302        0302
             0303        0303
             0304        0304

Saved inventory to: c:\Users\Admin\OneDrive\Documents\india-economic-complexity\data\interim\hs92_product_code_inventory.csv


In [8]:

from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
inventory_file = (
    project_root / "data" / "interim" /
    "hs92_product_code_inventory.csv"
)

inventory = pd.read_csv(
    inventory_file,
    dtype={"product_hs92_code": "string", "code_4digit": "string"}
)

# Provisional analytical grouping, not an official 22-sector classification.
sector_rules = [
    (1, 14, 1, "Agriculture, live animals and plant products"),
    (15, 24, 2, "Food, beverages, fats, oils and tobacco"),
    (25, 26, 3, "Mining and non-fuel mineral products"),
    (27, 27, 4, "Petroleum, coal and refined fuels"),
    (28, 29, 5, "Chemicals and petrochemicals"),
    (30, 30, 6, "Pharmaceuticals and medicinal products"),
    (31, 38, 5, "Chemicals and petrochemicals"),
    (39, 40, 7, "Plastics and rubber products"),
    (41, 43, 8, "Leather, hides, furs and related products"),
    (44, 49, 9, "Wood, pulp, paper and printing products"),
    (50, 63, 10, "Textiles, apparel and textile articles"),
    (64, 67, 11, "Footwear, headgear and related products"),
    (68, 70, 12, "Stone, ceramics, glass, cement and related products"),
    (71, 71, 13, "Precious metals, gems and jewellery"),
    (72, 83, 14, "Basic metals and fabricated metal products"),
    (84, 84, 15, "Industrial machinery and mechanical equipment"),
    (85, 85, 16, "Electrical machinery, electronics and electrical equipment"),
    (86, 89, 17, "Transport equipment"),
    (90, 92, 18, "Medical devices, optical and precision instruments"),
    (93, 93, 19, "Arms and ammunition"),
    (94, 99, 20, "Furniture, toys, miscellaneous manufactures and special goods"),
]

def map_sector(code):
    if pd.isna(code) or not str(code).isdigit():
        return pd.Series([pd.NA, "unmapped_special_code"])

    code = str(code).zfill(4)
    chapter = int(code[:2])

    for start, end, sector_id, sector_name in sector_rules:
        if start <= chapter <= end:
            return pd.Series([sector_id, "provisional_chapter_mapping"])

    return pd.Series([pd.NA, "unmapped_chapter"])

inventory[["sector_id", "mapping_status"]] = inventory[
    "code_4digit"
].apply(map_sector)

print("Total distinct codes:", len(inventory))
print("\nMapping status:")
print(inventory["mapping_status"].value_counts(dropna=False))

print("\nCodes requiring investigation:")
print(
    inventory.loc[
        inventory["mapping_status"].isin(
            ["unmapped_special_code", "unmapped_chapter"]
        ),
        ["product_hs92_code", "code_4digit", "mapping_status"]
    ].to_string(index=False)
)

print("\nNumeric codes mapped provisionally:",
      int(inventory["mapping_status"].eq("provisional_chapter_mapping").sum()))


Total distinct codes: 1243

Mapping status:
mapping_status
provisional_chapter_mapping    1242
unmapped_special_code             1
Name: count, dtype: int64

Codes requiring investigation:
product_hs92_code code_4digit        mapping_status
             XXXX        <NA> unmapped_special_code

Numeric codes mapped provisionally: 1242


In [9]:

from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
inventory_file = (
    project_root / "data" / "interim" /
    "hs92_product_code_inventory.csv"
)
output_file = (
    project_root / "data" / "interim" /
    "hs92_hs4_sector_mapping.csv"
)

inventory = pd.read_csv(
    inventory_file,
    dtype={"product_hs92_code": "string", "code_4digit": "string"}
)

sector_rules = [
    (1, 14, 1, "Agriculture, live animals and plant products"),
    (15, 24, 2, "Food, beverages, fats, oils and tobacco"),
    (25, 26, 3, "Mining and non-fuel mineral products"),
    (27, 27, 4, "Petroleum, coal and refined fuels"),
    (28, 29, 5, "Chemicals and petrochemicals"),
    (30, 30, 6, "Pharmaceuticals and medicinal products"),
    (31, 38, 5, "Chemicals and petrochemicals"),
    (39, 40, 7, "Plastics and rubber products"),
    (41, 43, 8, "Leather, hides, furs and related products"),
    (44, 49, 9, "Wood, pulp, paper and printing products"),
    (50, 63, 10, "Textiles, apparel and textile articles"),
    (64, 67, 11, "Footwear, headgear and related products"),
    (68, 70, 12, "Stone, ceramics, glass, cement and related products"),
    (71, 71, 13, "Precious metals, gems and jewellery"),
    (72, 83, 14, "Basic metals and fabricated metal products"),
    (84, 84, 15, "Industrial machinery and mechanical equipment"),
    (85, 85, 16, "Electrical machinery, electronics and electrical equipment"),
    (86, 89, 17, "Transport equipment"),
    (90, 92, 18, "Medical devices, optical and precision instruments"),
    (93, 93, 19, "Arms and ammunition"),
    (94, 99, 20, "Furniture, toys, miscellaneous manufactures and special goods"),
]

sector_lookup = {}
for start, end, sector_id, sector_name in sector_rules:
    for chapter in range(start, end + 1):
        sector_lookup[chapter] = (sector_id, sector_name)

rows = []

for _, row in inventory.iterrows():
    code = row["code_4digit"]

    if pd.isna(code) or not str(code).isdigit():
        rows.append({
            "product_hs92_code": row["product_hs92_code"],
            "hs_chapter": pd.NA,
            "sector_id": pd.NA,
            "sector_name": pd.NA,
            "mapping_status": "unmapped_special_code",
            "notes": "Investigate source meaning before assigning"
        })
        continue

    code = str(code).zfill(4)
    chapter = int(code[:2])
    sector = sector_lookup.get(chapter)

    if sector is None:
        rows.append({
            "product_hs92_code": row["product_hs92_code"],
            "hs_chapter": chapter,
            "sector_id": pd.NA,
            "sector_name": pd.NA,
            "mapping_status": "unmapped_chapter",
            "notes": "Chapter not covered by provisional rules"
        })
    else:
        sector_id, sector_name = sector
        rows.append({
            "product_hs92_code": row["product_hs92_code"],
            "hs_chapter": chapter,
            "sector_id": sector_id,
            "sector_name": sector_name,
            "mapping_status": "provisional_chapter_mapping",
            "notes": "Chapter-level assignment; validation pending"
        })

mapping = pd.DataFrame(rows)
mapping.to_csv(output_file, index=False)

print("Saved:", output_file)
print("\nMapping status:")
print(mapping["mapping_status"].value_counts(dropna=False))
print("\nPreview:")
display(mapping.head(10))


Saved: c:\Users\Admin\OneDrive\Documents\india-economic-complexity\data\interim\hs92_hs4_sector_mapping.csv

Mapping status:
mapping_status
provisional_chapter_mapping    1242
unmapped_special_code             1
Name: count, dtype: int64

Preview:


,product_hs92_code,hs_chapter,sector_id,sector_name,mapping_status,notes
0,0101,1,1,"Agriculture, live animals and plant products",provisional_chapter_mapping,Chapter-level assignment; validation pending
1,0102,1,1,"Agriculture, live animals and plant products",provisional_chapter_mapping,Chapter-level assignment; validation pending
2,0103,1,1,"Agriculture, live animals and plant products",provisional_chapter_mapping,Chapter-level assignment; validation pending
3,0104,1,1,"Agriculture, live animals and plant products",provisional_chapter_mapping,Chapter-level assignment; validation pending
4,0105,1,1,"Agriculture, live animals and plant products",provisional_chapter_mapping,Chapter-level assignment; validation pending
5,0106,1,1,"Agriculture, live animals and plant products",provisional_chapter_mapping,Chapter-level assignment; validation pending
6,0201,2,1,"Agriculture, live animals and plant products",provisional_chapter_mapping,Chapter-level assignment; validation pending
7,0202,2,1,"Agriculture, live animals and plant products",provisional_chapter_mapping,Chapter-level assignment; validation pending
8,0203,2,1,"Agriculture, live animals and plant products",provisional_chapter_mapping,Chapter-level assignment; validation pending
9,0204,2,1,"Agriculture, live animals and plant products",provisional_chapter_mapping,Chapter-level assignment; validation pending


In [10]:

from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
file = project_root / "data" / "interim" / "hs92_hs4_sector_mapping.csv"

mapping = pd.read_csv(
    file,
    dtype={"product_hs92_code": "string"}
)

print("Columns:", mapping.columns.tolist())
print("Rows:", len(mapping))
print("\nFirst 5 rows:")
print(mapping.head().to_string(index=False))

print("\nDuplicate product codes:",
      mapping["product_hs92_code"].duplicated().sum())

print("\nMissing sector assignments by status:")
print(
    mapping.groupby("mapping_status", dropna=False)["sector_id"]
    .apply(lambda s: s.isna().sum())
)

print("\nSector counts:")
print(
    mapping[mapping["sector_id"].notna()]
    .groupby(["sector_id", "sector_name"])["product_hs92_code"]
    .count()
    .to_string()
)

print("\nSpecial codes:")
print(
    mapping.loc[
        mapping["mapping_status"] != "provisional_chapter_mapping",
        ["product_hs92_code", "mapping_status", "notes"]
    ].to_string(index=False)
)


Columns: ['product_hs92_code', 'hs_chapter', 'sector_id', 'sector_name', 'mapping_status', 'notes']
Rows: 1243

First 5 rows:
product_hs92_code  hs_chapter  sector_id                                  sector_name              mapping_status                                        notes
             0101         1.0        1.0 Agriculture, live animals and plant products provisional_chapter_mapping Chapter-level assignment; validation pending
             0102         1.0        1.0 Agriculture, live animals and plant products provisional_chapter_mapping Chapter-level assignment; validation pending
             0103         1.0        1.0 Agriculture, live animals and plant products provisional_chapter_mapping Chapter-level assignment; validation pending
             0104         1.0        1.0 Agriculture, live animals and plant products provisional_chapter_mapping Chapter-level assignment; validation pending
             0105         1.0        1.0 Agriculture, live animals and plant pr

In [11]:

from pathlib import Path

project_root = Path.cwd().parent

for folder_name in ["data/raw", "data/interim", "docs"]:
    folder = project_root / folder_name
    print(f"\n--- {folder_name} ---")

    for file in sorted(folder.iterdir()):
        if file.is_file():
            print(file.name)



--- data/raw ---
hs92_country_product_year_4.csv
README.md

--- data/interim ---
hs92_hs4_sector_mapping.csv
hs92_product_code_inventory.csv
README.md

--- docs ---
README.md
sector_classification.md
sector_mapping_rules.csv


In [12]:
import pandas as pd

file_path = "../data/raw/HSCodeandDescription.xlsx"

excel = pd.ExcelFile(file_path)
print("Sheets:", excel.sheet_names)

for sheet in excel.sheet_names:
    sample = pd.read_excel(file_path, sheet_name=sheet, nrows=5)
    print(f"\nSheet: {sheet}")
    print("Columns:", sample.columns.tolist())
    print(sample.head(3).to_string(index=False))

Sheets: ['HS22', 'HS17', 'HS12', 'HS07', 'HS02', 'HS96', 'HS92']

Sheet: HS22
Columns: ['Classification', 'Code', 'Description', 'Parent Code', 'Level', 'IsBasicLevel']
Classification  Code                              Description Parent Code  Level  IsBasicLevel
            H6     1                            Animals; live       TOTAL      2             0
            H6   101   Horses, asses, mules and hinnies; live          01      4             0
            H6 10121 Horses; live, pure-bred breeding animals        0101      6             1

Sheet: HS17
Columns: ['Classification', 'Code', 'Description', 'Parent Code', 'Level', 'IsBasicLevel']
Classification  Code                              Description Parent Code  Level  IsBasicLevel
            H5     1                            Animals; live       TOTAL      2             0
            H5   101   Horses, asses, mules and hinnies; live          01      4             0
            H5 10121 Horses; live, pure-bred breeding animals 

In [13]:

import pandas as pd
from pathlib import Path

# Paths
input_file = Path("../data/raw/HSCodeandDescription.xlsx")
output_file = Path("../data/interim/hs92_hs4_descriptions.csv")

# Read the HS 1992 sheet
hs92 = pd.read_excel(input_file, sheet_name="HS92", dtype={"Code": str})

# Keep only the HS 1988/1992 classification
hs92 = hs92[hs92["Classification"] == "H0"].copy()

# Restore leading zeros according to each code's hierarchy level
hs92["Code"] = hs92.apply(
    lambda row: str(row["Code"]).strip().zfill(int(row["Level"])),
    axis=1
)

# Keep four-digit HS codes and their descriptions
hs4 = hs92[hs92["Level"] == 4][
    ["Code", "Description", "Parent Code", "Classification", "Level"]
].copy()

hs4 = hs4.rename(columns={
    "Code": "product_hs92_code",
    "Description": "product_description",
    "Parent Code": "hs_chapter"
})

# Check for duplicate codes before saving
print("HS4 rows:", len(hs4))
print("Unique HS4 codes:", hs4["product_hs92_code"].nunique())
print("Duplicate codes:", hs4["product_hs92_code"].duplicated().sum())
print("\nSample descriptions:")
print(hs4.head(10).to_string(index=False))

# Save the cleaned reference file
output_file.parent.mkdir(parents=True, exist_ok=True)
hs4.to_csv(output_file, index=False, encoding="utf-8")

print(f"\nSaved to: {output_file}")


HS4 rows: 1242
Unique HS4 codes: 1242
Duplicate codes: 0

Sample descriptions:
product_hs92_code                                                                           product_description hs_chapter Classification  Level
             0101                                                        Horses, asses, mules and hinnies; live         01             H0      4
             0102                                                                          Bovine animals; live         01             H0      4
             0103                                                                                   Swine; live         01             H0      4
             0104                                                                         Sheep and goats; live         01             H0      4
             0105 Poultry; live, fowls of the species gallus domesticus, ducks, geese, turkeys and guinea fowls         01             H0      4
             0106                                  

In [14]:

import pandas as pd
from pathlib import Path

raw_file = Path("../data/raw/hs92_country_product_year_4.csv")
desc_file = Path("../data/interim/hs92_hs4_descriptions.csv")

# Read only the product-code column from the large trade dataset
trade_codes = pd.read_csv(
    raw_file,
    usecols=["product_hs92_code"],
    dtype={"product_hs92_code": str}
)

# Normalize codes without changing the source file
trade_codes["product_hs92_code"] = (
    trade_codes["product_hs92_code"]
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

numeric_mask = trade_codes["product_hs92_code"].str.fullmatch(r"\d+")
trade_codes.loc[numeric_mask, "product_hs92_code"] = (
    trade_codes.loc[numeric_mask, "product_hs92_code"].str.zfill(4)
)

observed_codes = set(
    trade_codes.loc[numeric_mask, "product_hs92_code"].unique()
)

descriptions = pd.read_csv(
    desc_file,
    dtype={"product_hs92_code": str}
)
reference_codes = set(descriptions["product_hs92_code"])

matched = observed_codes & reference_codes
missing = sorted(observed_codes - reference_codes)

print("Unique numeric trade codes:", len(observed_codes))
print("Unique HS92 reference codes:", len(reference_codes))
print("Matched codes:", len(matched))
print("Unmatched numeric trade codes:", len(missing))
print("Coverage: {:.2f}%".format(
    100 * len(matched) / len(observed_codes)
))

if missing:
    print("\nFirst 30 unmatched codes:")
    print(missing[:30])
else:
    print("\nAll numeric trade codes have HS92 descriptions.")


Unique numeric trade codes: 1242
Unique HS92 reference codes: 1242
Matched codes: 1242
Unmatched numeric trade codes: 0
Coverage: 100.00%

All numeric trade codes have HS92 descriptions.


In [15]:

import pandas as pd
from pathlib import Path

inventory_file = Path("../data/interim/hs92_product_code_inventory.csv")
description_file = Path("../data/interim/hs92_hs4_descriptions.csv")
mapping_file = Path("../data/interim/hs92_hs4_sector_mapping.csv")
audit_file = Path("../data/interim/hs92_sector_mapping_audit.csv")

inventory = pd.read_csv(
    inventory_file,
    dtype={"product_hs92_code": str}
)
descriptions = pd.read_csv(
    description_file,
    dtype={"product_hs92_code": str}
)
mapping = pd.read_csv(
    mapping_file,
    dtype={"product_hs92_code": str}
)

# Join the three reference tables by the exact HS4 code
audit = inventory.merge(
    descriptions[["product_hs92_code", "product_description"]],
    on="product_hs92_code",
    how="left",
    validate="one_to_one"
)

audit = audit.merge(
    mapping[
        ["product_hs92_code", "sector_id", "sector_name",
         "mapping_status", "notes"]
    ],
    on="product_hs92_code",
    how="left",
    validate="one_to_one"
)

print("Audit rows:", len(audit))
print("Missing descriptions:", audit["product_description"].isna().sum())
print("Missing sector assignments:", audit["sector_id"].isna().sum())

print("\nFirst 30 product-level assignments:")
print(
    audit[
        ["product_hs92_code", "product_description",
         "sector_id", "sector_name", "mapping_status"]
    ].head(30).to_string(index=False)
)

audit.to_csv(audit_file, index=False, encoding="utf-8")
print(f"\nSaved audit file: {audit_file}")


Audit rows: 1243
Missing descriptions: 1
Missing sector assignments: 1

First 30 product-level assignments:
product_hs92_code                                                                                                                                                                                     product_description  sector_id                                  sector_name              mapping_status
             0101                                                                                                                                                                  Horses, asses, mules and hinnies; live        1.0 Agriculture, live animals and plant products provisional_chapter_mapping
             0102                                                                                                                                                                                    Bovine animals; live        1.0 Agriculture, live animals and plant products provisional_ch

In [16]:

import pandas as pd

audit = pd.read_csv(
    "../data/interim/hs92_sector_mapping_audit.csv",
    dtype={"product_hs92_code": str}
)

# Show records with missing descriptions or sector assignments
problems = audit[
    audit["product_description"].isna()
    | audit["sector_id"].isna()
]

print("Records needing attention:")
print(
    problems[
        ["product_hs92_code", "product_description",
         "sector_id", "sector_name", "mapping_status", "notes"]
    ].to_string(index=False)
)

# Check how food-related products are currently assigned
print("\nSelected food-related products:")
print(
    audit[
        audit["product_hs92_code"].str.startswith(
            ("02", "03", "04", "16", "17", "18", "19", "20", "21", "22")
        )
    ][
        ["product_hs92_code", "product_description",
         "sector_id", "sector_name"]
    ].head(25).to_string(index=False)
)


Records needing attention:
product_hs92_code product_description  sector_id sector_name        mapping_status                                       notes
             XXXX                 NaN        NaN         NaN unmapped_special_code Investigate source meaning before assigning

Selected food-related products:
product_hs92_code                                                                                                                                                                                     product_description  sector_id                                  sector_name
             0201                                                                                                                                                                Meat of bovine animals; fresh or chilled        1.0 Agriculture, live animals and plant products
             0202                                                                                                                         

In [17]:

import pandas as pd
from pathlib import Path

base = Path("../data/interim")
mapping_file = base / "hs92_hs4_sector_mapping.csv"
description_file = base / "hs92_hs4_descriptions.csv"

mapping = pd.read_csv(
    mapping_file,
    dtype={"product_hs92_code": str}
)
descriptions = pd.read_csv(
    description_file,
    dtype={"product_hs92_code": str}
)

# Attach descriptions for an auditable mapping
mapping = mapping.drop(
    columns=["product_description"], errors="ignore"
).merge(
    descriptions[["product_hs92_code", "product_description"]],
    on="product_hs92_code",
    how="left",
    validate="one_to_one"
)

# Derive the HS chapter from the first two digits
code = mapping["product_hs92_code"].fillna("").str.strip()
numeric_hs4 = code.str.fullmatch(r"\d{4}")

mapping["hs_chapter"] = pd.NA
mapping.loc[numeric_hs4, "hs_chapter"] = (
    code.loc[numeric_hs4].str[:2]
)

# Update the two affected sector assignments
animal_product_chapters = {"02", "03", "04", "05"}
sector_2_mask = mapping["hs_chapter"].isin(animal_product_chapters)

mapping.loc[sector_2_mask, "sector_id"] = 2
mapping.loc[sector_2_mask, "sector_name"] = (
    "Food and animal products, beverages, fats, oils and tobacco"
)

mapping.loc[
    mapping["hs_chapter"].isin({"01", *[f"{i:02d}" for i in range(6, 15)]}),
    "sector_id"
] = 1

mapping.loc[
    mapping["hs_chapter"].isin({"01", *[f"{i:02d}" for i in range(6, 15)]}),
    "sector_name"
] = "Agriculture, live animals and plant products"

# Rename existing Sector 2 entries in chapters 15–24 too
sector_2_all = mapping["sector_id"].eq(2)
mapping.loc[sector_2_all, "sector_name"] = (
    "Food and animal products, beverages, fats, oils and tobacco"
)

# Leave special codes such as XXXX unmapped
mapping.loc[~numeric_hs4, "sector_id"] = pd.NA
mapping.loc[~numeric_hs4, "sector_name"] = pd.NA
mapping.loc[~numeric_hs4, "mapping_status"] = "unmapped_special_code"

output_file = base / "hs92_hs4_sector_mapping_v2.csv"
mapping.to_csv(output_file, index=False, encoding="utf-8")

print("Saved:", output_file)
print("\nSector 1 and 2 counts:")
print(
    mapping[mapping["sector_id"].isin([1, 2])]
    .groupby(["sector_id", "sector_name"])
    .size()
    .to_string()
)

print("\nUnmapped codes:")
print(
    mapping.loc[
        mapping["sector_id"].isna(),
        ["product_hs92_code", "product_description", "mapping_status"]
    ].to_string(index=False)
)

print("\nTotal rows:", len(mapping))
print("Unique product codes:", mapping["product_hs92_code"].nunique())


Saved: ..\data\interim\hs92_hs4_sector_mapping_v2.csv

Sector 1 and 2 counts:
sector_id  sector_name                                                
1.0        Agriculture, live animals and plant products                    85
2.0        Food and animal products, beverages, fats, oils and tobacco    116

Unmapped codes:
product_hs92_code product_description        mapping_status
             XXXX                 NaN unmapped_special_code

Total rows: 1243
Unique product codes: 1243


In [18]:

import pandas as pd

mapping = pd.read_csv(
    "../data/interim/hs92_hs4_sector_mapping_v2.csv",
    dtype={"product_hs92_code": str}
)

# 1. Count codes per sector
counts = (
    mapping.dropna(subset=["sector_id"])
    .groupby(["sector_id", "sector_name"])
    .size()
    .reset_index(name="product_code_count")
    .sort_values("sector_id")
)

print("PRODUCT COUNTS BY SECTOR")
print(counts.to_string(index=False))

# 2. Show a few descriptions from every sector
print("\nSAMPLE PRODUCT ASSIGNMENTS")
for _, sector in counts.iterrows():
    subset = mapping[
        mapping["sector_id"].eq(sector["sector_id"])
    ].head(4)

    print(
        f"\nSector {int(sector['sector_id'])}: "
        f"{sector['sector_name']}"
    )
    for _, row in subset.iterrows():
        print(
            f"  {row['product_hs92_code']} — "
            f"{row['product_description']}"
        )

# 3. Save a compact review report
counts.to_csv(
    "../data/interim/hs92_sector_code_counts_v2.csv",
    index=False,
    encoding="utf-8"
)

print("\nSaved sector counts report.")


PRODUCT COUNTS BY SECTOR
 sector_id                                                   sector_name  product_code_count
       1.0                  Agriculture, live animals and plant products                  85
       2.0   Food and animal products, beverages, fats, oils and tobacco                 116
       3.0                          Mining and non-fuel mineral products                  51
       4.0                             Petroleum, coal and refined fuels                  16
       5.0                                  Chemicals and petrochemicals                 170
       6.0                        Pharmaceuticals and medicinal products                   6
       7.0                                  Plastics and rubber products                  43
       8.0                     Leather, hides, furs and related products                  21
       9.0                       Wood, pulp, paper and printing products                  68
      10.0                        Textiles, a

In [19]:

import pandas as pd

mapping = pd.read_csv(
    "../data/interim/hs92_hs4_sector_mapping_v2.csv",
    dtype={"product_hs92_code": str}
)

# Exclude special codes such as XXXX
numeric = mapping[
    mapping["product_hs92_code"].str.fullmatch(r"\d{4}", na=False)
].copy()

numeric["hs_chapter"] = numeric["product_hs92_code"].str[:2]

# Summarize each chapter's sector assignments
chapter_audit = (
    numeric.groupby("hs_chapter")
    .agg(
        product_codes=("product_hs92_code", "nunique"),
        number_of_sectors=("sector_id", "nunique"),
        sector_ids=("sector_id", lambda s: ", ".join(
            str(int(x)) for x in sorted(s.dropna().unique())
        )),
        sector_names=("sector_name", lambda s: " | ".join(
            sorted(s.dropna().unique())
        )),
        missing_sector_assignments=("sector_id", lambda s: s.isna().sum())
    )
    .reset_index()
    .sort_values("hs_chapter")
)

print("CHAPTER-LEVEL AUDIT")
print(chapter_audit.to_string(index=False))

# Flag chapters that need investigation
issues = chapter_audit[
    (chapter_audit["number_of_sectors"] != 1)
    | (chapter_audit["missing_sector_assignments"] > 0)
]

print("\nCHAPTERS NEEDING REVIEW:")
if issues.empty:
    print("No chapters are split across sectors or missing assignments.")
else:
    print(issues.to_string(index=False))

chapter_audit.to_csv(
    "../data/interim/hs92_chapter_mapping_audit.csv",
    index=False
)

print("\nSaved chapter audit.")


CHAPTER-LEVEL AUDIT
hs_chapter  product_codes  number_of_sectors sector_ids                                                  sector_names  missing_sector_assignments
        01              6                  1          1                  Agriculture, live animals and plant products                           0
        02             10                  1          2   Food and animal products, beverages, fats, oils and tobacco                           0
        03              7                  1          2   Food and animal products, beverages, fats, oils and tobacco                           0
        04             10                  1          2   Food and animal products, beverages, fats, oils and tobacco                           0
        05             11                  1          2   Food and animal products, beverages, fats, oils and tobacco                           0
        06              4                  1          1                  Agriculture, live animals and p

In [20]:
import pandas as pd

rules = pd.read_csv("../docs/sector_mapping_rules.csv")
print("Columns:", rules.columns.tolist())
print(rules.to_string(index=False))

print("\nDocumented sector definitions:")
with open("../docs/sector_classification.md", encoding="utf-8") as f:
    print(f.read())

Columns: ['hs_chapter_start', 'hs_chapter_end', 'sector_id', 'sector_name', 'mapping_status', 'notes']
 hs_chapter_start  hs_chapter_end  sector_id                                                   sector_name mapping_status                                   notes
                1              14          1                  Agriculture, live animals and plant products    provisional                Chapter-based assignment
               15              24          2                       Food, beverages, fats, oils and tobacco    provisional                Chapter-based assignment
               25              26          3                          Mining and non-fuel mineral products    provisional                Chapter-based assignment
               27              27          4                             Petroleum, coal and refined fuels    provisional                Chapter-based assignment
               28              29          5                                  Chemicals

In [1]:

import pandas as pd

rules = pd.read_csv("../docs/sector_mapping_rules.csv")
mapping = pd.read_csv(
    "../data/interim/hs92_hs4_sector_mapping_v2.csv",
    dtype={"product_hs92_code": str}
)

# Keep only numeric HS4 codes and derive the HS chapter.
numeric = mapping[
    mapping["product_hs92_code"].str.fullmatch(r"\d{1,4}", na=False)
].copy()

numeric["code_4digit"] = numeric["product_hs92_code"].str.zfill(4)
numeric["hs_chapter"] = numeric["code_4digit"].str[:2].astype(int)

def assign_sector(chapter):
    match = rules[
        (rules["hs_chapter_start"] <= chapter) &
        (rules["hs_chapter_end"] >= chapter)
    ]
    if len(match) != 1:
        return None
    return int(match.iloc[0]["sector_id"])

numeric["sector_from_rules"] = numeric["hs_chapter"].apply(assign_sector)

mismatches = numeric[
    numeric["sector_from_rules"] != numeric["sector_id"]
]

print("Numeric HS4 codes checked:", len(numeric))
print("Unassigned by rules:", numeric["sector_from_rules"].isna().sum())
print("Mismatches with mapping v2:", len(mismatches))

if len(mismatches):
    print(mismatches[
        ["product_hs92_code", "hs_chapter", "sector_id", "sector_from_rules"]
    ].head(20).to_string(index=False))
else:
    print("PASS: The documented rules reproduce mapping v2.")


Numeric HS4 codes checked: 1242
Unassigned by rules: 0
Mismatches with mapping v2: 0
PASS: The documented rules reproduce mapping v2.


In [2]:

import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

raw_file = ROOT / "data/raw/hs92_country_product_year_4.csv"
mapping_file = ROOT / "data/interim/hs92_hs4_sector_mapping_v2.csv"
output_file = ROOT / "data/processed/country_year_sector_trade.csv"

# Load the saved sector totals and mapping.
sector_trade = pd.read_csv(output_file)
mapping = pd.read_csv(
    mapping_file,
    dtype={"product_hs92_code": str}
)

print("Saved output columns:")
print(sector_trade.columns.tolist())

print("\nChecking for duplicate country-year-sector records...")
keys = ["country_iso3_code", "year", "sector_id"]
print("Duplicate rows:", sector_trade.duplicated(keys).sum())

print("\nChecking export and import totals against source...")
raw = pd.read_csv(
    raw_file,
    dtype={"product_hs92_code": str},
    low_memory=False
)

raw["product_hs92_code"] = raw["product_hs92_code"].str.zfill(4)
mapping["product_hs92_code"] = mapping["product_hs92_code"].str.zfill(4)

raw = raw.merge(
    mapping[["product_hs92_code", "sector_id"]],
    on="product_hs92_code",
    how="left",
    validate="many_to_one"
)

mapped_raw = raw.dropna(subset=["sector_id"])

for column in ["export_value", "import_value"]:
    source_total = mapped_raw[column].sum()
    aggregated_total = sector_trade[column].sum()
    difference = aggregated_total - source_total

    print(f"\n{column}:")
    print(f"Mapped source total: {source_total:,.2f}")
    print(f"Sector dataset total: {aggregated_total:,.2f}")
    print(f"Difference: {difference:,.6f}")
    print("PASS" if abs(difference) < 0.01 else "CHECK REQUIRED")


Saved output columns:
['country_id', 'country_iso3_code', 'year', 'sector_id', 'sector_name', 'export_value', 'import_value', 'product_count', 'trade_balance']

Checking for duplicate country-year-sector records...
Duplicate rows: 0

Checking export and import totals against source...

export_value:
Mapped source total: 378,037,574,862,003.00
Sector dataset total: 378,037,574,862,003.00
Difference: 0.000000
PASS

import_value:
Mapped source total: 378,037,574,862,003.00
Sector dataset total: 378,037,574,862,003.00
Difference: 0.000000
PASS


In [3]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

df = pd.read_csv(
    ROOT / "data/raw/hs92_country_product_year_4.csv",
    dtype={"product_hs92_code": str},
    low_memory=False
)

india = df[
    (df["country_iso3_code"] == "IND") &
    (df["year"] == df["year"].max())
].copy()

print("Latest year:", india["year"].max())
print("India product records:", len(india))
print("Unique HS4 codes:", india["product_hs92_code"].nunique())
print("Total exports:", india["export_value"].sum())
print("Missing RCA:", india["export_rca"].isna().sum())
print("Missing PCI:", india["pci"].isna().sum())
print("\nSample:")
print(
    india[
        ["product_hs92_code", "export_value", "export_rca", "pci"]
    ].head(10).to_string(index=False)
)

Latest year: 2024
India product records: 1238
Unique HS4 codes: 1238
Total exports: 416457390569
Missing RCA: 0
Missing PCI: 0

Sample:
product_hs92_code  export_value  export_rca    pci
             0101         62243    0.000799  0.496
             0102             0    0.000000 -0.531
             0103          4753    0.000046  1.129
             0104      18448658    0.461621 -1.487
             0105       1094500    0.014762 -0.460
             0106        689341    0.023262 -1.230
             0201     115879594    0.175665 -0.358
             0202    3496042843    4.776934 -1.093
             0203        798870    0.001077  0.559
             0204      84970517    0.474709 -1.361
